In [9]:
# LLM은 최신 정보를 모른다.
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke("로제가 발표한 신곡은 무엇인가요?")

AIMessage(content='로제는 최근 "On The Ground"와 "GONE"이라는 곡을 발표했습니다. 두 곡은 그녀의 첫 솔로 앨범인 "R"의 일부로, 각각 다양한 감정과 주제를 다루고 있습니다. 더 최신 소식이나 다른 곡에 대한 정보는 공식 팬사이트나 음악 스트리밍 플랫폼을 확인해 보시는 것이 좋습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 18, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5ffc63ae9', 'id': 'chatcmpl-EZS4sIiyMK788U11AmWutqTSvcQZx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a14290-5728-7100-94f8-47a9a562136c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 83, 'total_t

In [2]:
# %pip install ddgs

In [10]:
# client("로제가 발표한 신곡은 무엇인가요?") => llm => 2023년 10월 기준 => client
# client의 질문 => ddgs => 검색결과 => 질문+검색결과 => llm => 최신 내용(답변) => client
from langchain_community.tools import DuckDuckGoSearchResults   # 검색어를 받아 검색 결과 처리

# 질문 => ddgs
search = DuckDuckGoSearchResults(results_separator=";\n")   # 검색 결과 여러 개 나오는 것의 구분자 지정.

docs = search.invoke("로제 최신 노래 알려줘.")
print(docs)

snippet: 그러니까 자기야, 네 결정을 알려줘 Do I call you my ex 너를 전 연인이라고 부를까, Or do I call you my boyfriend 아니면 내 남자친구라고 부를까? Call you a lover, do I call you a friend 내 사랑이라고 불러야 할지, 아니면 그냥 친구라고 해야 할지 Call you the one or the one that got away, title: rosie - 나무위키, link: https://namu.wiki/w/rosie;
snippet: 로제 (ROSÉ) - new trick [가사/Lyrics] #로제 #ROSÉ #newtrick 발매 : 2026.09.18 아티스트 : 로제 (ROSÉ) 앨범 : new trick 작곡 : 로제 (ROSÉ), Amy Allen 작사 : 로제 (ROSÉ ..., title: 로제 (ROSÉ) - new trick [가사/Lyrics] - YouTube, link: https://www.youtube.com/watch?v=wjHsyPPL-Os;
snippet: ROSÉ • new trick Released: 09.17.2026 ℗ 2026 ROSÉ under exclusive license to Atlantic Recording Corporation for the World excluding Korea #ROSÉ #로제 #newtrick #ROSÉ_newtrick …...more, title: ROSÉ 로제 'new trick' Official Audio - YouTube, link: https://www.youtube.com/watch?v=gkuHUlAm1OM;
snippet: 글로벌 아이콘 블랙핑크의 멤버 로제 (ROSÉ)가 통통 튀는 에너지와 당찬 매력으로 무장한 신곡 'new trick (뉴 트릭)'을 선보입니다. 글로벌 메가 히트를 기록한 'APT.' 이후 발표하는 이번 신곡은 음악적 완성도는 물론, 세계적인 브랜드와의 협업으로 공개 전

In [11]:
# 질문 -> ddgs -> docs(검색결과) + 질문 -> LLM
from langchain_core.prompts import ChatPromptTemplate   # 질문 + 검색결과 저장 반복 사용하는 틀
from langchain_core.prompts import MessagesPlaceholder  # 질문을 저장하는 틀

# 질문 + 검색 결과 저장 틀 생성 => 변수 대입
question_answering_prompt = ChatPromptTemplate.from_messages(   # 리스트: 여러 개 저장받겠다.
    [
        (
            "system",
            "아래 context에 기반하여 사용자의 질문에 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | llm    # 질문 + 검색결과 => llm 연결만 해 놓은 상태.


In [12]:
# 대화 내용을 저장
from langchain_community.chat_message_histories import ChatMessageHistory   # 대화(질문, 답변) 내용 저장.

chat_history = ChatMessageHistory()     # 대화 저장할 수 있는 저장소 생성
# 질문 생성 => 저장소 저장
chat_history.add_user_message("최근 로제가 발표한 신곡은 무엇인가요?")

# 질문을 llm에 보내고 답변 받기. : document_chain
answer = document_chain.invoke(
    {
        "context":docs,                     # 검색 결과
        "messages":chat_history.messages    # 질문과 답변
    }
)
# answer 기능: 
#   1. 화면 출력
#   2. 답변 저장 후 chat_history에 저장해야 멀티턴 가능.
chat_history.add_ai_message(answer)
print(answer)

content="로제가 최근 발표한 신곡은 'new trick'입니다. 이 곡은 2026년 9월 17일에 발매되었으며, 통통 튀는 에너지와 매력을 담고 있습니다. 'new trick'은 새로운 방법이나 묘수를 의미하는 제목을 가지고 있습니다." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 563, 'total_tokens': 630, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9cf9d5c9b2', 'id': 'chatcmpl-EZS51QIFrGqLmtgZGRHlErVBJigiX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a14290-7b07-7cd2-9237-329b9a2ce720-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 563, 'output_tokens': 67, 'total_tokens': 630, 'input_token_details': {'audio': 0

In [20]:
# 필터링
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

ddgs_wrapper = DuckDuckGoSearchAPIWrapper(
    region="kr-kr", 
    time="w"
)


In [21]:
search = DuckDuckGoSearchResults(
    api_wrapper=ddgs_wrapper,   # 한국, 일주일치 지정
    source="news",              # 뉴스에서만 검색하도록 지정
    results_separator=";\n"     # 검색 결과 항목 사이에 구분자 지정
)

In [23]:
docs = search.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

print(docs)

snippet: 2,935 likes, 39 comments - billboard_korea on October 14, 2026: "로제가 LACMA 아트+필름 갈라 스페셜 퍼포먼스로 선다! 11월 7일, LA 카운티 미술관에서 열리는 제15회 갈라에서 음악 게스트로 무대를 꾸며요. 2022년, 2023년 LACMA 레드카펫에 게스트로 섰던 로제가 2026년엔 무대 위 주인공으로 돌아와요 어떤 곡을 들려줄지 ..., title: Billboard Korea | 빌보드코리아 | 로제가 LACMA 아트+필름 갈라 스페셜 ..., link: https://www.instagram.com/p/DeeHp2lFPs7/;
snippet: 특히 이 곡은 로제가 자신의 첫 정규 앨범 작업 과정에서 가장 진솔한 감정을 담아낸 곡 중 하나로 알려지며 더욱 화제를 모았습니다. 인터뷰에서는 앨범 전체가 자신의 실제 경험과 감정에서 출발했다고 밝히기도 했는데요., title: 뮤맥 | ROSÉ의 toxic till the end는 사랑이 끝난 후에도 쉽게 지워지지 ..., link: https://www.instagram.com/reel/DeZOiLmui9-/;
snippet: 블랙핑크 로제가 다음 달 7일 미국 로스앤젤레스에서 열리는 2026 '아트+필름 갈라'에서 공연을 펼칩니다. '아트 필름 갈라'는 로스앤젤레스 카운티 ..., title: 로제, 디카프리오 주최 '아트+필름 갈라'서 공연, link: https://news.sbs.co.kr/news/endPage.do?news_id=N1008798217;
snippet: 특히 이 곡은 로제가 자신의 첫 정규 앨범 작업 과정에서 가장 진솔한 감정을 담아낸 곡 중 하나로 알려지며 더욱 화제를 모았습니다. 인터뷰에서는 앨범 전체가 자신의 실제 경험과 감정에서 출발했다고 밝히기도 했는데요. 그래서인지 많은 팬들이 "지금까지 로제의 노래 중 가장 솔직한 이야기 같다"는 반응을 보였습니다., title: 핍 뮤직 • 노래 추천 | ROS

In [29]:
try:
    docs_0 = search.invoke("site:ytn.co.kr 최근 로제가 발표한 신곡은 무엇인가요?")

    print(docs_0)
except Exception as e:
    print(e)
print("검색 정상 종료")

No results found.
검색 정상 종료


In [31]:
print(docs)

snippet: 2,935 likes, 39 comments - billboard_korea on October 14, 2026: "로제가 LACMA 아트+필름 갈라 스페셜 퍼포먼스로 선다! 11월 7일, LA 카운티 미술관에서 열리는 제15회 갈라에서 음악 게스트로 무대를 꾸며요. 2022년, 2023년 LACMA 레드카펫에 게스트로 섰던 로제가 2026년엔 무대 위 주인공으로 돌아와요 어떤 곡을 들려줄지 ..., title: Billboard Korea | 빌보드코리아 | 로제가 LACMA 아트+필름 갈라 스페셜 ..., link: https://www.instagram.com/p/DeeHp2lFPs7/;
snippet: 특히 이 곡은 로제가 자신의 첫 정규 앨범 작업 과정에서 가장 진솔한 감정을 담아낸 곡 중 하나로 알려지며 더욱 화제를 모았습니다. 인터뷰에서는 앨범 전체가 자신의 실제 경험과 감정에서 출발했다고 밝히기도 했는데요., title: 뮤맥 | ROSÉ의 toxic till the end는 사랑이 끝난 후에도 쉽게 지워지지 ..., link: https://www.instagram.com/reel/DeZOiLmui9-/;
snippet: 블랙핑크 로제가 다음 달 7일 미국 로스앤젤레스에서 열리는 2026 '아트+필름 갈라'에서 공연을 펼칩니다. '아트 필름 갈라'는 로스앤젤레스 카운티 ..., title: 로제, 디카프리오 주최 '아트+필름 갈라'서 공연, link: https://news.sbs.co.kr/news/endPage.do?news_id=N1008798217;
snippet: 특히 이 곡은 로제가 자신의 첫 정규 앨범 작업 과정에서 가장 진솔한 감정을 담아낸 곡 중 하나로 알려지며 더욱 화제를 모았습니다. 인터뷰에서는 앨범 전체가 자신의 실제 경험과 감정에서 출발했다고 밝히기도 했는데요. 그래서인지 많은 팬들이 "지금까지 로제의 노래 중 가장 솔직한 이야기 같다"는 반응을 보였습니다., title: 핍 뮤직 • 노래 추천 | ROS

In [33]:
len(docs.split(";\n"))

4

In [35]:
# 기사 원문 링크 저장
links = []  # 링크 저장하는 리스트

for doc in docs.split(";\n") :  # 현재 기사가 4개다. 4번 반복
    # print(doc)
    link=doc.split("link:")[1].strip()
    links.append(link)

print(links)

['https://www.instagram.com/p/DeeHp2lFPs7/', 'https://www.instagram.com/reel/DeZOiLmui9-/', 'https://news.sbs.co.kr/news/endPage.do?news_id=N1008798217', 'https://www.instagram.com/reel/DeZPIhoyzi4/']
